# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [52]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [53]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['HMQBCIAEPF', 'VVAWKZZJYI'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 8, 13, 17,  2,  3,  9,  1,  5, 16,  6],
       [22, 22,  1, 23, 11, 26, 26, 10, 25,  9]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  6, 16,  5,  1,  9,  3,  2, 17, 13],
       [ 0,  9, 25, 10, 26, 26, 11, 23,  1, 22]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 6, 16,  5,  1,  9,  3,  2, 17, 13,  8],
       [ 9, 25, 10, 26, 26, 11, 23,  1, 22, 22]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [ ]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        todo
        
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        # 编码器输出和初始状态
        enc_out, state = self.encode(enc_ids)   # state: [last_output, last_state]
        # 预先计算编码器输出的投影，用于双线性注意力
        enc_proj = self.dense_attn(enc_out)     # (batch, enc_len, hidden)

        # 解码器输入嵌入
        dec_emb = self.embed_layer(dec_ids)     # (batch, dec_len, emb_sz)

        # 初始化解码器状态（取编码器最后一步的隐藏状态）
        init_state = state[1] if isinstance(state, (list, tuple)) else state

        # 手动逐步解码，以便在每个时间步插入注意力
        logits_list = []
        decoder_state = init_state  # 单个张量 (batch, hidden)

        for t in range(dec_ids.shape[1]):  # 遍历每个解码时间步
            # 当前时间步的输入
            x_t = dec_emb[:, t, :]          # (batch, emb_sz)

            # 通过解码器 RNN cell
            output, decoder_state = self.decoder_cell(x_t, [decoder_state])  # output 和 new_state 形状 (batch, hidden)
            output = output[0] if isinstance(output, (list, tuple)) else output  # 确保是张量
            decoder_state = decoder_state[0] if isinstance(decoder_state, (list, tuple)) else decoder_state

            # 双线性注意力：score = output^T * W * enc_out
            # 利用 dense_attn 对 enc_out 做线性变换，然后点积
            scores = tf.reduce_sum(output[:, tf.newaxis, :] * enc_proj, axis=-1)  # (batch, enc_len)
            attn_weights = tf.nn.softmax(scores, axis=-1)                         # (batch, enc_len)
            context = tf.reduce_sum(attn_weights[..., tf.newaxis] * enc_out, axis=1)  # (batch, hidden)

            # 结合 RNN 输出和上下文向量，预测下一个词
            combined = tf.concat([output, context], axis=-1)   # (batch, hidden*2)
            logits = self.dense(combined)                      # (batch, v_sz)
            logits_list.append(logits)

        # 堆叠所有时间步的输出
        logits = tf.stack(logits_list, axis=1)  # (batch, dec_len, v_sz)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
    
        '''
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑'''
        # 1. 对输入token进行embedding
        x = tf.cast(x, tf.int32)
        x_emb = self.embed_layer(x)  # (batch, emb_sz)
        
        # 2. 通过RNN cell
        if isinstance(state, (list, tuple)):
            actual_state = state[0] if len(state) > 0 else state
        else:
            actual_state = state
        
        if isinstance(actual_state, (list, tuple)):
            actual_state = actual_state[0]
        
        # RNN cell 需要状态作为列表
        h, new_state = self.decoder_cell(x_emb, [actual_state])
        h = h[0] if isinstance(h, (list, tuple)) else h
        new_state = new_state[0] if isinstance(new_state, (list, tuple)) else new_state
        
        # 3. 双线性注意力机制
        # 将encoder输出投影到与decoder隐藏状态相同的空间
        enc_proj = self.dense_attn(enc_out)  # (batch, enc_len, hidden)
        
        # 计算注意力分数: 投影后的enc_out与decoder隐藏状态的点积
        scores = tf.reduce_sum(enc_proj * tf.expand_dims(h, axis=1), axis=-1)  # (batch, enc_len)
        
        # 计算注意力权重
        alpha = tf.nn.softmax(scores, axis=-1)  # (batch, enc_len)
        
        # 计算上下文向量
        context = tf.reduce_sum(enc_out * tf.expand_dims(alpha, axis=-1), axis=1)  # (batch, hidden)
        
        # 4. 结合解码器隐藏状态和上下文向量
        combined = tf.concat([h, context], axis=-1)  # (batch, hidden*2)
        
        # 5. 通过输出层得到logits
        logits = self.dense(combined)  # (batch, v_sz)
        
        # 6. 获取预测的token（概率最大的索引）
        next_token = tf.argmax(logits, axis=-1)  # (batch,)
        
        return next_token, new_state

# Loss函数以及训练逻辑

In [55]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [56]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.2953873
step 500 : loss 1.2958617
step 1000 : loss 0.1203831
step 1500 : loss 0.05210497


<tf.Tensor: shape=(), dtype=float32, numpy=0.029638994>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [57]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('JJUQUVRTTWDFOCDJKKYT', 'TYKKJDCOFDWTTRVUQUJP'), ('GWEGWKQCBAYNFEZKGONK', 'KNOGKZEFNYABCQKWGEWM'), ('DWIVYPCURAEUGWJTHATH', 'NTAHTJWGUEARUCPYVIWD'), ('DURLSGZMGDRTFWDORJDK', 'KDJRODWFTRDGMZGSLRUD'), ('AOPWVHXRRVGSNSIGERUK', 'KUREGISNSGVRRXHVWPOA'), ('CCFHFYXGVFLPVFUSODJQ', 'QJDOSUFVPLFVGXYFHFCC'), ('RYUWJOIHAKRECXELCGFN', 'NFGCLEXCERKAHIOJWUYR'), ('BXRFBATRHWHZMLFAAVAA', 'AAVAAFLMZHWHRTABFRXB'), ('MUKNPEYHGHGJDXUIZPMT', 'TMPZIUXDJGHGHYEPNKUM'), ('MCXFGBKUMGMHXPVXFACI', 'ICAFXVPXHMGMUKBGFXCM'), ('ZUCUSYLTVVFNCUYAQQXD', 'DXQQAYUCNFVVTLYSUCUZ'), ('DKUOIBVHPUCCTWGZOEOG', 'GOEOZGWTCCUPHVBIOUKD'), ('LDUAKQIFJDOUNXRDRUOJ', 'JOURDRXNUODJFIQKAUDL'), ('AHWSVIPMYOWEJAEHNEKL', 'LKENHEAJEWOYMPIVSWHA'), ('YRGGYGEYGCDHCDDSOMRD', 'DRMOSDDCHDCGYEGYGGOA'), ('JXYDWGFYFTOKXCLSBACB', 'BCABSLCXKOTFYFGWDYXJ'), ('KPKR